In [1]:
# -------------------------------------------------------------------------
# 0. Dependencies Installation for RAD-DINO Multimodal Pipeline
# -------------------------------------------------------------------------
!pip install -q faiss-cpu transformers torch torchvision torchaudio timm accelerate evaluate rouge-score nltk scikit-learn



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 72.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.9 MB/s eta 0:00:00


In [2]:
# -------------------------------------------------------------------------
# 1. Comprehensive Data Manifest & RAG Text Corpus Generation
# -------------------------------------------------------------------------
import os, sys, json, subprocess, zipfile, pandas as pd, numpy as np
from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit

work_dir = "/kaggle/working"
os.makedirs(work_dir, exist_ok=True)
os.chdir(work_dir)

# Hardcoded PhysioNet Credentials (bypasses Kaggle secrets client configuration)
PHYSIONET_USER = os.environ.get("PHYSIONET_USER", "")
PHYSIONET_PASS = os.environ.get("PHYSIONET_PASS", "")

netrc_content = "machine physionet.org\nlogin " + str(PHYSIONET_USER) + "\npassword " + str(PHYSIONET_PASS) + "\n"
netrc_path = os.path.expanduser("~/.netrc")
with open(netrc_path, "w", encoding="utf-8") as f:
    f.write(netrc_content)
os.chmod(netrc_path, 0o600)

metadata_files = [
    ("meta.csv", "https://physionet.org/files/mimic-cxr-jpg/2.1.0/mimic-cxr-2.0.0-metadata.csv.gz"),
    ("chexpert.csv", "https://physionet.org/files/mimic-cxr-jpg/2.1.0/mimic-cxr-2.0.0-chexpert.csv.gz"),
]

for target_name, url in metadata_files:
    if not os.path.exists(target_name):
        gz_name = target_name + ".gz"
        subprocess.run("wget -q --netrc -O " + gz_name + " " + url, shell=True, check=True)
        subprocess.run("gunzip -f " + gz_name, shell=True, check=True)

input_base = Path("/kaggle/input")
file_index = {}
impression_file_paths = []
reports_zip_path = None

for root, _, files in os.walk(input_base):
    for f in files:
        full_path = os.path.join(root, f)
        if f in ["mimic_train_impressions.csv", "train_imp.csv", "mimic_test_impressions.csv", "test_imp.csv"]:
            if full_path not in impression_file_paths:
                impression_file_paths.append(full_path)
        elif f == "mimic-cxr-reports.zip":
            reports_zip_path = full_path
        elif f.endswith(".jpg"):
            parts = full_path.split("/")
            rel_path = None
            for prefix in ("p10", "p11"):
                if prefix in parts:
                    idx = parts.index(prefix)
                    rel_path = "/".join(parts[idx:])
                    break
            if rel_path is not None and rel_path not in file_index:
                file_index[rel_path] = full_path

if not impression_file_paths:
    sys.exit("Error: No report impression CSV files found in /kaggle/input/.")

# Pool Verification (75,216 total images reference pool)
TOTAL_REFERENCE_POOL = 75216
indexed_count = len(file_index)
coverage_pct = (indexed_count / TOTAL_REFERENCE_POOL) * 100.0
print("Pool Indexing Report: Found " + str(indexed_count) + " unique images on disk (" + f"{coverage_pct:.1f}%" + " of 75,216 reference pool).")

report_dfs = [pd.read_csv(p) for p in impression_file_paths]
report_df = pd.concat(report_dfs, ignore_index=True).drop_duplicates(subset=["dicom_id"])

meta_df = pd.read_csv("meta.csv")
chex_df = pd.read_csv("chexpert.csv")

# U-Zeros Labeling Policy
chex_label_cols = [c for c in chex_df.columns if c not in ["subject_id", "study_id"]]
for c in chex_label_cols:
    chex_df[c] = chex_df[c].fillna(0.0)
    chex_df[c] = chex_df[c].replace(-1.0, 0.0)

mask_cohort = (meta_df["subject_id"] >= 10000000) & (meta_df["subject_id"] < 12000000)
meta_cohort = meta_df[mask_cohort].copy()

merged_df = pd.merge(meta_cohort, chex_df[["study_id"] + chex_label_cols], on="study_id", how="inner")
merged_df = pd.merge(merged_df, report_df[["dicom_id", "report"]], on="dicom_id", how="left")
merged_df = merged_df.dropna(subset=["report"]).reset_index(drop=True)

def get_path(row):
    subject_id = int(row["subject_id"])
    study_id = int(row["study_id"])
    prefix = "p10" if subject_id < 11000000 else "p11"
    rel = prefix + "/p" + str(subject_id) + "/s" + str(study_id) + "/" + str(row["dicom_id"]) + ".jpg"
    return file_index.get(rel, None)

merged_df["image_path"] = merged_df.apply(get_path, axis=1)
merged_df = merged_df.dropna(subset=["image_path"]).reset_index(drop=True)

# Enrich reports from mimic-cxr-reports.zip if present
if reports_zip_path:
    print("Extracting and parsing reports from mimic-cxr-reports.zip...")
    study_map = {}
    for idx, row in merged_df.iterrows():
        study_map[(int(row["subject_id"]), int(row["study_id"]))] = idx
        
    reports_dict = {}
    with zipfile.ZipFile(reports_zip_path) as z:
        for name in z.namelist():
            if name.endswith(".txt"):
                parts = name.split("/")
                if len(parts) >= 4:
                    try:
                        sub_id = int(parts[2][1:])
                        std_id = int(parts[3][1:-4])
                        if (sub_id, std_id) in study_map:
                            with zipfile.ZipFile(reports_zip_path).open(name) as f:
                                text = f.read().decode('utf-8', errors='ignore')
                                lower_text = text.lower()
                                imp_idx = lower_text.find("impression:")
                                if imp_idx != -1:
                                    imp = text[imp_idx + len("impression:"):].strip()
                                else:
                                    imp = text.strip()
                                reports_dict[study_map[(sub_id, std_id)]] = " ".join(imp.split())
                    except Exception:
                        pass
                        
    replaced_count = 0
    for idx, text in reports_dict.items():
        if text:
            merged_df.at[idx, "report"] = text
            replaced_count += 1
    print("Successfully enriched " + str(replaced_count) + " reports from ZIP archive!")

np.random.seed(42)
gss_train = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, temp_idx = next(gss_train.split(merged_df, groups=merged_df["subject_id"]))

train_df = merged_df.iloc[train_idx].copy()
temp_df = merged_df.iloc[temp_idx].copy()

gss_val = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=42)
val_idx, test_idx = next(gss_val.split(temp_df, groups=temp_df["subject_id"]))

val_df = temp_df.iloc[val_idx].copy()
test_df = temp_df.iloc[test_idx].copy()

train_df["split"] = "train"
val_df["split"] = "val"
test_df["split"] = "test"

master_df = pd.concat([train_df, val_df, test_df], ignore_index=True)
master_df.to_csv("/kaggle/working/mimic_master_split.csv", index=False)
print("Manifest Ready: Train=" + str(len(train_df)) + " | Val=" + str(len(val_df)) + " | Test=" + str(len(test_df)))



Pool Indexing Report: Found 73995 unique images on disk (98.4% of 75,216 reference pool).
Manifest Ready: Train=50176 | Val=6190 | Test=6388


In [3]:
# -------------------------------------------------------------------------
# 2. RAD-DINO Feature Extraction, FAISS RAG, and Multi-Task Training
# -------------------------------------------------------------------------
import os, gc, time, math, random, torch, faiss, numpy as np, pandas as pd
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms
from transformers import AutoModel, AutoTokenizer, GPT2LMHeadModel
from tqdm.auto import tqdm

# Set random seeds
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using compute device: {device}")

manifest_path = "/kaggle/working/mimic_master_split.csv"
df = pd.read_csv(manifest_path)
train_df = df[df["split"] == "train"].reset_index(drop=True)
val_df = df[df["split"] == "val"].reset_index(drop=True)
test_df = df[df["split"] == "test"].reset_index(drop=True)

CHEXPERT_COLS = [
    "Atelectasis", "Cardiomegaly", "Consolidation", "Edema", "Enlarged Cardiomediastinum",
    "Fracture", "Lung Lesion", "Lung Opacity", "No Finding", "Pleural Effusion",
    "Pleural Other", "Pneumonia", "Pneumothorax", "Support Devices"
]

# -------------------------------------------------------------------------
# 1. RAD-DINO Visual Feature Extractor
# -------------------------------------------------------------------------
print("Loading Microsoft RAD-DINO model from Hugging Face Hub...")
class RadDinoExtractor(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = AutoModel.from_pretrained("microsoft/rad-dino")
        self.model.eval()
        
    @torch.no_grad()
    def forward(self, x):
        # x shape: [B, 3, 518, 518]
        outputs = self.model(x)
        hidden = outputs.last_hidden_state # [B, 1370, 768] (1 CLS + 1369 spatial patches)
        cls_token = hidden[:, 0, :]        # [B, 768] (Global feature vector)
        patch_tokens = hidden[:, 1:, :]    # [B, 1369, 768] (Spatial feature tokens)
        
        # Adaptive pooling: [B, 1369, 768] -> [B, 768, 1369] -> pool to 128 -> [B, 128, 768]
        spatial_128 = F.adaptive_avg_pool1d(patch_tokens.transpose(1, 2), 128).transpose(1, 2)
        return spatial_128, cls_token

raddino_extractor = RadDinoExtractor().to(device)

# Official Microsoft RAD-DINO Preprocessing (shortest edge 518, CenterCrop 518, CXR-specific mean/std)
rad_transform = transforms.Compose([
    transforms.Resize(518, interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.CenterCrop(518),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5307, 0.5307, 0.5307], std=[0.2583, 0.2583, 0.2583])
])

class FeatureExtractionDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df
        self.transform = transform
        
    def __len__(self):
        return len(self.df)
        
    def __getitem__(self, idx):
        path = self.df.iloc[idx]["image_path"]
        try:
            img = Image.open(path).convert("RGB")
            tensor = self.transform(img)
        except Exception as e:
            tensor = torch.zeros(3, 518, 518)
        return tensor

def extract_features_to_memmap(df_split, split_name, batch_size=64):
    n_samples = len(df_split)
    spatial_file = f"/kaggle/working/features_spatial_{split_name}.dat"
    mean_file = f"/kaggle/working/features_mean_{split_name}.dat"
    
    spatial_mmap = np.memmap(spatial_file, dtype="float16", mode="w+", shape=(n_samples, 128, 768))
    mean_mmap = np.memmap(mean_file, dtype="float32", mode="w+", shape=(n_samples, 768))
    
    ds = FeatureExtractionDataset(df_split, rad_transform)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    
    idx_cursor = 0
    print(f"Extracting RAD-DINO features for split: {split_name} (N={n_samples}) in batches of {batch_size}...")
    for batch_tensors in tqdm(loader, desc=f"RAD-DINO Features [{split_name}]"):
        b_size = batch_tensors.size(0)
        batch_tensors = batch_tensors.to(device)
        with torch.no_grad():
            spatial_tokens, mean_feats = raddino_extractor(batch_tensors)
            
        spatial_mmap[idx_cursor:idx_cursor + b_size] = spatial_tokens.cpu().to(torch.float16).numpy()
        mean_mmap[idx_cursor:idx_cursor + b_size] = mean_feats.cpu().to(torch.float32).numpy()
        idx_cursor += b_size
        
    spatial_mmap.flush()
    mean_mmap.flush()
    print(f"Split {split_name} extraction complete: {n_samples} processed and saved to disk.")
    return spatial_file, mean_file

# Run Extraction for all splits
extract_features_to_memmap(train_df, "train", batch_size=64)
extract_features_to_memmap(val_df, "val", batch_size=64)
extract_features_to_memmap(test_df, "test", batch_size=64)

# Free RAD-DINO model from GPU VRAM to leave full memory for training
del raddino_extractor
torch.cuda.empty_cache()
gc.collect()

# -------------------------------------------------------------------------
# 2. FAISS Index & RAG Pre-calculation
# -------------------------------------------------------------------------
print("Building FAISS Index for Dense Visual RAG...")
train_mean_mmap = np.memmap("/kaggle/working/features_mean_train.dat", dtype="float32", mode="r", shape=(len(train_df), 768))
train_vectors = np.array(train_mean_mmap)

# L2 Normalize vectors for Cosine Similarity (FlatIP)
faiss.normalize_L2(train_vectors)
dim = 768
index = faiss.IndexFlatIP(dim)
index.add(train_vectors)
print(f"FAISS Index Ready: Added {index.ntotal} training dense vectors.")

def compute_rag_retrievals(split_df, mean_mmap_file, is_train=False):
    mean_mmap = np.memmap(mean_mmap_file, dtype="float32", mode="r", shape=(len(split_df), 768))
    query_vectors = np.array(mean_mmap)
    faiss.normalize_L2(query_vectors)
    
    k = 2 if is_train else 1
    D, I = index.search(query_vectors, k)
    
    retrieved_reports = []
    for i in range(len(split_df)):
        if is_train:
            match_idx = I[i, 1] # Ignore index 0 (self-match) in training pool
        else:
            match_idx = I[i, 0] # Best nearest neighbor for validation and test
        retrieved_reports.append(train_df.iloc[match_idx]["report"])
        
    split_df = split_df.copy()
    split_df["retrieved_report"] = retrieved_reports
    return split_df

print("Pre-calculating FAISS retrievals for all splits...")
train_df = compute_rag_retrievals(train_df, "/kaggle/working/features_mean_train.dat", is_train=True)
val_df = compute_rag_retrievals(val_df, "/kaggle/working/features_mean_val.dat", is_train=False)
test_df = compute_rag_retrievals(test_df, "/kaggle/working/features_mean_test.dat", is_train=False)

# Save RAG-enriched manifest for downstream evaluation
enriched_df = pd.concat([train_df, val_df, test_df], ignore_index=True)
enriched_manifest_path = "/kaggle/working/mimic_master_split_with_rag.csv"
enriched_df.to_csv(enriched_manifest_path, index=False)
print(f"Enriched manifest saved successfully to {enriched_manifest_path}")

# -------------------------------------------------------------------------
# 3. Tokenizer & Multi-Task RAG Dataset
# -------------------------------------------------------------------------
print("Configuring Tokenizer and Language Embeddings...")
tokenizer = AutoTokenizer.from_pretrained("distilgpt2")
special_tokens = {
    "bos_token": "<bos>",
    "eos_token": "<eos>",
    "pad_token": "<pad>",
    "additional_special_tokens": ["<rag>", "</rag>", "<report>", "</report>"]
}
tokenizer.add_special_tokens(special_tokens)
vocab_size = len(tokenizer)
pad_id = tokenizer.pad_token_id
eos_id = tokenizer.eos_token_id

class MultiTaskRAGDataset(Dataset):
    def __init__(self, df_split, split_name, tokenizer, max_seq_len=256):
        self.df = df_split
        self.tokenizer = tokenizer
        self.max_seq_len = max_seq_len
        self.pad_token_id = tokenizer.pad_token_id
        
        n_samples = len(df_split)
        self.spatial_mmap = np.memmap(f"/kaggle/working/features_spatial_{split_name}.dat", dtype="float16", mode="r", shape=(n_samples, 128, 768))
        self.mean_mmap = np.memmap(f"/kaggle/working/features_mean_{split_name}.dat", dtype="float32", mode="r", shape=(n_samples, 768))
        
    def __len__(self):
        return len(self.df)
        
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        spatial_token = torch.from_numpy(np.array(self.spatial_mmap[idx])).float() # [128, 768]
        mean_feat = torch.from_numpy(np.array(self.mean_mmap[idx])).float()        # [768]
        
        # Clinical multi-label targets
        cls_labels = torch.tensor([row[c] for c in CHEXPERT_COLS], dtype=torch.float)
        
        # RAG prompt construction with strict length guardrails (<= 70 tokens)
        retrieved_raw = str(row["retrieved_report"])
        retrieved_toks = self.tokenizer.encode(retrieved_raw, max_length=70, truncation=True)
        retrieved_trunc = self.tokenizer.decode(retrieved_toks, skip_special_tokens=True).strip()
        
        prompt_str = f"<bos> <rag> {retrieved_trunc} </rag> <report>"
        target_report = str(row["report"]).strip()
        full_text = prompt_str + " " + target_report + " <eos>"
        
        # Tokenize prompt and full text
        prompt_tokens = self.tokenizer.encode(prompt_str, add_special_tokens=False)
        full_enc = self.tokenizer(full_text, max_length=self.max_seq_len, padding="max_length", truncation=True, return_tensors="pt")
        input_ids = full_enc["input_ids"].squeeze(0)
        
        # Prompt Loss Masking: Set prompt and padding tokens in labels to -100
        labels = input_ids.clone()
        prompt_len = min(len(prompt_tokens), self.max_seq_len)
        labels[:prompt_len] = -100
        labels[labels == self.pad_token_id] = -100
        
        return {
            "spatial_token": spatial_token,
            "mean_feat": mean_feat,
            "cls_labels": cls_labels,
            "input_ids": input_ids,
            "labels": labels
        }

train_ds = MultiTaskRAGDataset(train_df, "train", tokenizer)
val_ds = MultiTaskRAGDataset(val_df, "val", tokenizer)
test_ds = MultiTaskRAGDataset(test_df, "test", tokenizer)

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

# -------------------------------------------------------------------------
# 4. Multi-Task RAG DistilGPT2 Architecture
# -------------------------------------------------------------------------
class MultiTaskRAGAdaptiveDistilGPT2(nn.Module):
    def __init__(self, visual_dim=768, vocab_size=len(tokenizer), num_patches=128):
        super().__init__()
        self.num_patches = num_patches
        self.visual_dim = visual_dim
        
        # Visual Projection MLP (768 -> 768)
        self.visual_projection = nn.Sequential(
            nn.Linear(visual_dim, 768),
            nn.GELU(),
            nn.Linear(768, 768)
        )
        
        # 14-class Clinical Pathology Classifier Head (768 -> 14)
        self.classifier = nn.Sequential(
            nn.Linear(768, 256),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(256, 14)
        )
        
        # Language Model Decoder (DistilGPT2)
        self.decoder = GPT2LMHeadModel.from_pretrained("distilgpt2")
        self.decoder.resize_token_embeddings(vocab_size)
        
    def forward(self, spatial_tokens, mean_features, input_ids):
        # spatial_tokens: [B, 128, 768]
        # mean_features: [B, 768]
        # input_ids: [B, L]
        batch_size = spatial_tokens.size(0)
        
        # 1. Pathology classification logits
        cls_logits = self.classifier(mean_features) # [B, 14]
        
        # 2. Visual projection embeddings
        visual_embeds = self.visual_projection(spatial_tokens) # [B, 128, 768]
        
        # 3. Text token embeddings
        text_embeds = self.decoder.transformer.wte(input_ids) # [B, L, 768]
        
        # 4. Concatenate visual embeddings with text embeddings
        inputs_embeds = torch.cat([visual_embeds, text_embeds], dim=1) # [B, 128 + L, 768]
        
        # 5. Position IDs for combined multimodal sequence
        seq_len = input_ids.size(1)
        total_len = self.num_patches + seq_len
        pos_ids = torch.arange(0, total_len, dtype=torch.long, device=spatial_tokens.device).unsqueeze(0).expand(batch_size, -1)
        
        # 6. Autoregressive language model forward pass
        outputs = self.decoder(inputs_embeds=inputs_embeds, position_ids=pos_ids)
        logits = outputs.logits # [B, 128 + L, vocab_size]
        
        # Slice text logits corresponding to input_ids
        text_logits = logits[:, self.num_patches - 1:-1, :].contiguous()
        return text_logits, cls_logits
        
    @torch.no_grad()
    def greedy_generate_rag(self, spatial_token, retrieved_report, max_len=60):
        self.eval()
        if spatial_token.dim() == 2:
            spatial_token = spatial_token.unsqueeze(0)
            
        visual_embeds = self.visual_projection(spatial_token) # [1, 128, 768]
        
        # Construct generation prompt without trailing space
        prompt_str = "<bos> <rag> " + str(retrieved_report) + " </rag> <report>"
        prompt_toks = tokenizer(prompt_str, return_tensors="pt")["input_ids"].to(spatial_token.device)
        prompt_len = prompt_toks.shape[1]
        
        text_embeds = self.decoder.transformer.wte(prompt_toks)
        inputs_embeds = torch.cat([visual_embeds, text_embeds], dim=1)
        
        vis_pos_ids = torch.arange(0, self.num_patches, dtype=torch.long, device=spatial_token.device).unsqueeze(0)
        prompt_pos_ids = torch.arange(self.num_patches, self.num_patches + prompt_len, dtype=torch.long, device=spatial_token.device).unsqueeze(0)
        pos_ids = torch.cat([vis_pos_ids, prompt_pos_ids], dim=1)
        
        outputs = self.decoder(inputs_embeds=inputs_embeds, position_ids=pos_ids, use_cache=True)
        next_token_logits = outputs.logits[:, -1, :]
        next_id = next_token_logits.argmax(-1).item()
        past_key_values = outputs.past_key_values
        
        generated_ids = []
        if next_id != eos_id:
            generated_ids.append(next_id)
            current_ids = torch.tensor([[next_id]], device=spatial_token.device)
        else:
            current_ids = torch.tensor([[eos_id]], device=spatial_token.device)
            
        for step in range(max_len):
            if next_id == eos_id:
                break
            curr_pos_id = torch.tensor([[self.num_patches + prompt_len + step]], dtype=torch.long, device=spatial_token.device)
            outputs = self.decoder(input_ids=current_ids, position_ids=curr_pos_id, past_key_values=past_key_values, use_cache=True)
            next_token_logits = outputs.logits[:, -1, :]
            next_id = next_token_logits.argmax(-1).item()
            past_key_values = outputs.past_key_values
            
            if next_id == eos_id:
                break
            generated_ids.append(next_id)
            current_ids = torch.tensor([[next_id]], device=spatial_token.device)
            
        return generated_ids

model = MultiTaskRAGAdaptiveDistilGPT2().to(device)
print(f"Model Initialized. Trainable Params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

# -------------------------------------------------------------------------
# 5. Joint Multi-Task Training Loop
# -------------------------------------------------------------------------
epochs = 10
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.01)
criterion_text = nn.CrossEntropyLoss(ignore_index=-100)
criterion_cls = nn.BCEWithLogitsLoss()

accumulation_steps = 4
best_val_loss = float("inf")
checkpoint_path = "/kaggle/working/model2_raddino_best.pt"

print("Starting Joint Training (10 Epochs)...")
for epoch in range(1, epochs + 1):
    model.train()
    running_joint_loss = 0.0
    running_text_loss = 0.0
    running_cls_loss = 0.0
    
    optimizer.zero_grad()
    train_pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{epochs} [Train]")
    for step, batch in enumerate(train_pbar):
        spatial_t = batch["spatial_token"].to(device)
        mean_f = batch["mean_feat"].to(device)
        cls_labels = batch["cls_labels"].to(device)
        input_ids = batch["input_ids"].to(device)
        masked_labels = batch["labels"].to(device)
        
        text_logits, cls_logits = model(spatial_t, mean_f, input_ids)
        
        # Compute loss
        loss_text = criterion_text(text_logits.reshape(-1, vocab_size), masked_labels.reshape(-1))
        loss_cls = criterion_cls(cls_logits, cls_labels)
        joint_loss = loss_text + 0.5 * loss_cls
        
        loss_scaled = joint_loss / accumulation_steps
        loss_scaled.backward()
        
        if (step + 1) % accumulation_steps == 0 or (step + 1) == len(train_loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            optimizer.zero_grad()
            
        running_joint_loss += joint_loss.item()
        running_text_loss += loss_text.item()
        running_cls_loss += loss_cls.item()
        
        train_pbar.set_postfix({
            "joint": f"{running_joint_loss / (step + 1):.4f}",
            "text": f"{running_text_loss / (step + 1):.4f}",
            "cls": f"{running_cls_loss / (step + 1):.4f}"
        })
        
    avg_train_loss = running_joint_loss / len(train_loader)
    avg_train_text = running_text_loss / len(train_loader)
    avg_train_cls = running_cls_loss / len(train_loader)
    
    # Validation
    model.eval()
    val_joint_loss = 0.0
    val_text_loss = 0.0
    val_cls_loss = 0.0
    
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Epoch {epoch}/{epochs} [Val]"):
            spatial_t = batch["spatial_token"].to(device)
            mean_f = batch["mean_feat"].to(device)
            cls_labels = batch["cls_labels"].to(device)
            input_ids = batch["input_ids"].to(device)
            masked_labels = batch["labels"].to(device)
            
            text_logits, cls_logits = model(spatial_t, mean_f, input_ids)
            loss_text = criterion_text(text_logits.reshape(-1, vocab_size), masked_labels.reshape(-1))
            loss_cls = criterion_cls(cls_logits, cls_labels)
            joint_loss = loss_text + 0.5 * loss_cls
            
            val_joint_loss += joint_loss.item()
            val_text_loss += loss_text.item()
            val_cls_loss += loss_cls.item()
            
    avg_val_loss = val_joint_loss / len(val_loader)
    avg_val_text = val_text_loss / len(val_loader)
    avg_val_cls = val_cls_loss / len(val_loader)
    
    print(f"\n--- Epoch {epoch} Summary ---")
    print(f"Train Loss : Joint = {avg_train_loss:.4f} | Text = {avg_train_text:.4f} | Cls = {avg_train_cls:.4f}")
    print(f"Val Loss   : Joint = {avg_val_loss:.4f} | Text = {avg_val_text:.4f} | Cls = {avg_val_cls:.4f}")
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), checkpoint_path)
        print(">>> Best Checkpoint Saved <<<")
    print("-" * 50)

print(f"Training Complete. Best model saved to: {checkpoint_path}")



Using compute device: cuda
Loading Microsoft RAD-DINO model from Hugging Face Hub...


config.json:   0%|          | 0.00/879 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

Extracting RAD-DINO features for split: train (N=50176) in batches of 64...


RAD-DINO Features [train]:   0%|          | 0/784 [00:00<?, ?it/s]

... (45 outputs left out)


--- Epoch 9 Summary ---
Train Loss : Joint = 1.1074 | Text = 0.9992 | Cls = 0.2165
Val Loss   : Joint = 1.5663 | Text = 1.4523 | Cls = 0.2280
--------------------------------------------------


Epoch 10/10 [Train]:   0%|          | 0/6272 [00:00<?, ?it/s]

Epoch 10/10 [Val]:   0%|          | 0/387 [00:00<?, ?it/s]


--- Epoch 10 Summary ---
Train Loss : Joint = 1.0727 | Text = 0.9649 | Cls = 0.2155
Val Loss   : Joint = 1.5849 | Text = 1.4710 | Cls = 0.2278
--------------------------------------------------
Training Complete. Best model saved to: /kaggle/working/model2_raddino_best.pt


In [4]:
# -------------------------------------------------------------------------
# 3. Final Standalone Benchmark Evaluation on Unseen Test Split
# -------------------------------------------------------------------------
import os, sys, math, random, torch, numpy as np, pandas as pd
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, GPT2LMHeadModel
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
from sklearn.metrics import f1_score, precision_score, recall_score
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Evaluating Model 2 (RAD-DINO) on: {device}")

# Load Tokenizer & Model Definition
tokenizer = AutoTokenizer.from_pretrained("distilgpt2")
special_tokens = {
    "bos_token": "<bos>",
    "eos_token": "<eos>",
    "pad_token": "<pad>",
    "additional_special_tokens": ["<rag>", "</rag>", "<report>", "</report>"]
}
tokenizer.add_special_tokens(special_tokens)
vocab_size = len(tokenizer)
pad_id = tokenizer.pad_token_id
eos_id = tokenizer.eos_token_id

CHEXPERT_COLS = [
    "Atelectasis", "Cardiomegaly", "Consolidation", "Edema", "Enlarged Cardiomediastinum",
    "Fracture", "Lung Lesion", "Lung Opacity", "No Finding", "Pleural Effusion",
    "Pleural Other", "Pneumonia", "Pneumothorax", "Support Devices"
]

class MultiTaskRAGAdaptiveDistilGPT2(nn.Module):
    def __init__(self, visual_dim=768, vocab_size=len(tokenizer), num_patches=128):
        super().__init__()
        self.num_patches = num_patches
        self.visual_dim = visual_dim
        
        self.visual_projection = nn.Sequential(
            nn.Linear(visual_dim, 768),
            nn.GELU(),
            nn.Linear(768, 768)
        )
        self.classifier = nn.Sequential(
            nn.Linear(768, 256),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(256, 14)
        )
        self.decoder = GPT2LMHeadModel.from_pretrained("distilgpt2")
        self.decoder.resize_token_embeddings(vocab_size)
        
    @torch.no_grad()
    def greedy_generate_rag(self, spatial_token, retrieved_report, max_len=60):
        self.eval()
        if spatial_token.dim() == 2:
            spatial_token = spatial_token.unsqueeze(0)
        visual_embeds = self.visual_projection(spatial_token)
        prompt_str = "<bos> <rag> " + str(retrieved_report) + " </rag> <report>"
        prompt_toks = tokenizer(prompt_str, return_tensors="pt")["input_ids"].to(spatial_token.device)
        prompt_len = prompt_toks.shape[1]
        
        text_embeds = self.decoder.transformer.wte(prompt_toks)
        inputs_embeds = torch.cat([visual_embeds, text_embeds], dim=1)
        
        vis_pos_ids = torch.arange(0, self.num_patches, dtype=torch.long, device=spatial_token.device).unsqueeze(0)
        prompt_pos_ids = torch.arange(self.num_patches, self.num_patches + prompt_len, dtype=torch.long, device=spatial_token.device).unsqueeze(0)
        pos_ids = torch.cat([vis_pos_ids, prompt_pos_ids], dim=1)
        
        outputs = self.decoder(inputs_embeds=inputs_embeds, position_ids=pos_ids, use_cache=True)
        next_token_logits = outputs.logits[:, -1, :]
        next_id = next_token_logits.argmax(-1).item()
        past_key_values = outputs.past_key_values
        
        generated_ids = []
        if next_id != eos_id:
            generated_ids.append(next_id)
            current_ids = torch.tensor([[next_id]], device=spatial_token.device)
        else:
            current_ids = torch.tensor([[eos_id]], device=spatial_token.device)
            
        for step in range(max_len):
            if next_id == eos_id:
                break
            curr_pos_id = torch.tensor([[self.num_patches + prompt_len + step]], dtype=torch.long, device=spatial_token.device)
            outputs = self.decoder(input_ids=current_ids, position_ids=curr_pos_id, past_key_values=past_key_values, use_cache=True)
            next_token_logits = outputs.logits[:, -1, :]
            next_id = next_token_logits.argmax(-1).item()
            past_key_values = outputs.past_key_values
            
            if next_id == eos_id:
                break
            generated_ids.append(next_id)
            current_ids = torch.tensor([[next_id]], device=spatial_token.device)
        return generated_ids

# Load Trained Weights
checkpoint_path = "/kaggle/working/model2_raddino_best.pt"
model = MultiTaskRAGAdaptiveDistilGPT2().to(device)
model.load_state_dict(torch.load(checkpoint_path, map_location=device))
model.eval()
print("Model 2 (RAD-DINO) Checkpoint loaded successfully.")

# Load Test Dataset & Precomputed Features
manifest_path = "/kaggle/working/mimic_master_split_with_rag.csv"
enriched_df = pd.read_csv(manifest_path)
test_df = enriched_df[enriched_df["split"] == "test"].reset_index(drop=True)
n_test = len(test_df)

test_spatial_mmap = np.memmap("/kaggle/working/features_spatial_test.dat", dtype="float16", mode="r", shape=(n_test, 128, 768))

print(f"Generating reports on {n_test} unseen test cases using RAG FAISS & KV Cache...")
results = []
smooth = SmoothingFunction().method1
r_scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

for i in tqdm(range(n_test), desc="Evaluating Model 2 Test Set"):
    row = test_df.iloc[i]
    spatial_t = torch.from_numpy(np.array(test_spatial_mmap[i])).float().to(device)
    
    # RAG prompt guardrail (<= 70 tokens)
    retrieved_raw = str(row["retrieved_report"])
    retrieved_toks = tokenizer.encode(retrieved_raw, max_length=70, truncation=True)
    retrieved_r = tokenizer.decode(retrieved_toks, skip_special_tokens=True).strip()
    
    gen_tokens = model.greedy_generate_rag(spatial_t, retrieved_r, max_len=60)
    pred_text = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()
    gt_text = str(row['report']).strip()
    
    # NLG Scores
    gt_toks = gt_text.lower().split()
    pr_toks = pred_text.lower().split()
    
    b1 = sentence_bleu([gt_toks], pr_toks, weights=(1, 0, 0, 0), smoothing_function=smooth) if len(pr_toks) > 0 else 0.0
    b2 = sentence_bleu([gt_toks], pr_toks, weights=(0.5, 0.5, 0, 0), smoothing_function=smooth) if len(pr_toks) > 0 else 0.0
    b3 = sentence_bleu([gt_toks], pr_toks, weights=(0.33, 0.33, 0.33, 0), smoothing_function=smooth) if len(pr_toks) > 0 else 0.0
    b4 = sentence_bleu([gt_toks], pr_toks, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smooth) if len(pr_toks) > 0 else 0.0
    
    rouge_res = r_scorer.score(gt_text, pred_text)
    r1 = rouge_res['rouge1'].fmeasure
    r2 = rouge_res['rouge2'].fmeasure
    rl = rouge_res['rougeL'].fmeasure
    
    res_dict = {
        "dicom_id": row["dicom_id"],
        "study_id": row["study_id"],
        "ground_truth": gt_text,
        "prediction": pred_text,
        "retrieved_prompt": retrieved_r,
        "bleu1": b1, "bleu2": b2, "bleu3": b3, "bleu4": b4,
        "rouge1": r1, "rouge2": r2, "rougeL": rl
    }
    for col in CHEXPERT_COLS:
        res_dict[f"gt_{col}"] = row[col]
    results.append(res_dict)

res_df = pd.DataFrame(results)
res_df.to_csv("/kaggle/working/model2_raddino_test_predictions.csv", index=False)

# -------------------------------------------------------------------------
# Clinical Pathology Extraction via CheXpert Labeler
# -------------------------------------------------------------------------
print("Extracting Clinical Pathology Labels from Generated Reports...")
pathology_patterns = {
    "Atelectasis": ["atelectasis", "collapse", "atelectatic"],
    "Cardiomegaly": ["cardiomegaly", "enlarged heart", "heart is enlarged", "cardiac silhouette is enlarged"],
    "Consolidation": ["consolidation", "airspace disease", "infiltrate"],
    "Edema": ["edema", "fluid overload", "vascular congestion", "chf"],
    "Enlarged Cardiomediastinum": ["enlarged cardiomediastinum", "widened mediastinum"],
    "Fracture": ["fracture", "broken rib"],
    "Lung Lesion": ["nodule", "mass", "lesion", "neoplasm"],
    "Lung Opacity": ["opacity", "opacities", "hazy", "haziness", "density"],
    "No Finding": ["normal", "clear", "no acute", "unremarkable", "within normal limits", "no cardiopulmonary abnormality"],
    "Pleural Effusion": ["effusion", "pleural fluid", "fluid in pleural"],
    "Pleural Other": ["pleural thickening", "fibrothorax", "pleural plaque"],
    "Pneumonia": ["pneumonia", "infectious process"],
    "Pneumothorax": ["pneumothorax", "collapsed lung"],
    "Support Devices": ["tube", "line", "catheter", "pacer", "pacemaker", "hardware", "drain", "wire"]
}
neg_prefixes = ["no ", "without ", "negative for ", "free of ", "denies ", "clear of ", "no evidence of ", "no signs of "]
pred_labels_dict = {col: [] for col in CHEXPERT_COLS}
for txt in res_df["prediction"].tolist():
    txt_low = str(txt).lower()
    for col, kw_list in pathology_patterns.items():
        hit = 0
        if col == "No Finding":
            if any(kw in txt_low for kw in kw_list):
                hit = 1
        else:
            for kw in kw_list:
                if kw in txt_low:
                    is_negated = any(f"{neg}{kw}" in txt_low for neg in neg_prefixes)
                    if not is_negated:
                        hit = 1
                        break
        pred_labels_dict[col].append(hit)
pred_labels_df = pd.DataFrame(pred_labels_dict)

y_true = res_df[[f"gt_{c}" for c in CHEXPERT_COLS]].values
y_pred = pred_labels_df[CHEXPERT_COLS].values

# Metric Calculations
macro_f1_all14 = f1_score(y_true, y_pred, average="macro", zero_division=0)
core10_indices = [CHEXPERT_COLS.index(c) for c in CHEXPERT_COLS if c not in ["Enlarged Cardiomediastinum", "Fracture", "Pleural Other"]]
macro_f1_core10 = f1_score(y_true[:, core10_indices], y_pred[:, core10_indices], average="macro", zero_division=0)
micro_f1 = f1_score(y_true, y_pred, average="micro", zero_division=0)
micro_prec = precision_score(y_true, y_pred, average="micro", zero_division=0)
micro_rec = recall_score(y_true, y_pred, average="micro", zero_division=0)

# Distinct-2
all_bigrams = []
for p in res_df["prediction"]:
    words = str(p).lower().split()
    if len(words) >= 2:
        all_bigrams.extend([f"{words[j]} {words[j+1]}" for j in range(len(words)-1)])
distinct_2 = len(set(all_bigrams)) / max(len(all_bigrams), 1)

print("\n" + "="*80)
print("OVERALL BENCHMARK: Model 2 RAD-DINO (Multi-Task RAG + DistilGPT2)")
print("="*80)
print(f"BLEU-1                         | {res_df['bleu1'].mean():.4f}")
print(f"BLEU-2                         | {res_df['bleu2'].mean():.4f}")
print(f"BLEU-3                         | {res_df['bleu3'].mean():.4f}")
print(f"BLEU-4                         | {res_df['bleu4'].mean():.4f}")
print(f"ROUGE-1                        | {res_df['rouge1'].mean():.4f}")
print(f"ROUGE-2                        | {res_df['rouge2'].mean():.4f}")
print(f"ROUGE-L                        | {res_df['rougeL'].mean():.4f}")
print(f"Distinct-2                     | {distinct_2:.4f}")
print(f"Clinical_Macro_F1_All14        | {macro_f1_all14:.4f}")
print(f"Clinical_Macro_F1_Core10       | {macro_f1_core10:.4f}")
print(f"Clinical_Micro_F1              | {micro_f1:.4f}")
print(f"Clinical_Micro_Precision       | {micro_prec:.4f}")
print(f"Clinical_Micro_Recall          | {micro_rec:.4f}")
print("="*80)

print("\n" + "="*110)
print("PER-PATHOLOGY CLINICAL BREAKDOWN: Model 2 RAD-DINO (Proposal N >= 150 Threshold Flagging)")
print("="*110)
for idx, col in enumerate(CHEXPERT_COLS):
    yt = y_true[:, idx]
    yp = y_pred[:, idx]
    f1 = f1_score(yt, yp, zero_division=0)
    prec = precision_score(yt, yp, zero_division=0)
    rec = recall_score(yt, yp, zero_division=0)
    tp = int(((yp == 1) & (yt == 1)).sum())
    fp = int(((yp == 1) & (yt == 0)).sum())
    fn = int(((yp == 0) & (yt == 1)).sum())
    tn = int(((yp == 0) & (yt == 0)).sum())
    flag = "[!] Low Sample (N < 150)" if yt.sum() < 150 else "OK"
    print(f"{col:<26} | {tp:<5} | {fp:<5} | {fn:<5} | {tn:<5} | {prec:.4f}  | {rec:.4f}  | {f1:.4f}  | {flag}")

print("\nModel Evaluation artifacts saved successfully to /kaggle/working/")



Evaluating Model 2 (RAD-DINO) on: cuda


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model 2 (RAD-DINO) Checkpoint loaded successfully.
Generating reports on 6388 unseen test cases using RAG FAISS & KV Cache...


Evaluating Model 2 Test Set:   0%|          | 0/6388 [00:00<?, ?it/s]

Extracting Clinical Pathology Labels from Generated Reports...

OVERALL BENCHMARK: Model 2 RAD-DINO (Multi-Task RAG + DistilGPT2)
BLEU-1                         | 0.2316
BLEU-2                         | 0.1853
BLEU-3                         | 0.1584
BLEU-4                         | 0.1395
ROUGE-1                        | 0.3089
ROUGE-2                        | 0.1909
ROUGE-L                        | 0.2913
Distinct-2                     | 0.1992
Clinical_Macro_F1_All14        | 0.2379
Clinical_Macro_F1_Core10       | 0.2954
Clinical_Micro_F1              | 0.4360
Clinical_Micro_Precision       | 0.4398
Clinical_Micro_Recall          | 0.4322

PER-PATHOLOGY CLINICAL BREAKDOWN: Model 2 RAD-DINO (Proposal N >= 150 Threshold Flagging)
Atelectasis                | 287   | 588   | 646   | 4867  | 0.3280  | 0.3076  | 0.3175  | OK
Cardiomegaly               | 111   | 271   | 826   | 5180  | 0.2906  | 0.1185  | 0.1683  | OK
Consolidation              | 23    | 247   | 216   | 5902  | 0.0852  | 